In [320]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
print(df["split"].value_counts())

Dataset shape: (12515, 24)
split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [321]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

all_vital_features = spo2_hr_features + [
    "respiratory_rate",
    "rr_lag_1",
    "rr_rolling_mean_5",
    "mean_arterial_pressure",
    "map_lag_1",
    "map_rolling_mean_5"
]

target = "future_spo2_5min"

In [322]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

common_val_df = val_df.dropna(
    subset=[
        "spo2",
        target
    ]
).copy()

train_ready = train_df.dropna(
    subset=[target]
).copy()

print("Training rows:", len(train_ready))
print("Validation rows:", len(common_val_df))

Training rows: 8310
Validation rows: 1748


In [323]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [324]:
def evaluate_random_forest(
    feature_list,
    feature_name
):

    X_train = train_ready[feature_list]
    y_train = train_ready[target]

    X_val = common_val_df[feature_list]
    y_val = common_val_df[target]

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    max_depth=10,
                    min_samples_leaf=3,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_val
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predictions
        )
    )

    r2 = r2_score(
        y_val,
        predictions
    )

    result = {
        "Model": feature_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    return model, predictions, result

In [325]:
rf_spo2_model, rf_spo2_pred, rf_spo2_result = (
    evaluate_random_forest(
        spo2_features,
        "Random Forest - SpO2 History"
    )
)

rf_spo2_result

{'Model': 'Random Forest - SpO2 History',
 'MAE': 0.42595033981559677,
 'RMSE': np.float64(1.4152561658650287),
 'R2': 0.04966757580330916}

In [326]:
rf_hr_model, rf_hr_pred, rf_hr_result = (
    evaluate_random_forest(
        spo2_hr_features,
        "Random Forest - SpO2 + HR"
    )
)

rf_hr_result

{'Model': 'Random Forest - SpO2 + HR',
 'MAE': 0.4452517629529656,
 'RMSE': np.float64(1.4365797977560144),
 'R2': 0.020814563962056387}

In [327]:
rf_all_model, rf_all_pred, rf_all_result = (
    evaluate_random_forest(
        all_vital_features,
        "Random Forest - All Vitals"
    )
)

rf_all_result

{'Model': 'Random Forest - All Vitals',
 'MAE': 0.4729700834642134,
 'RMSE': np.float64(1.4358696904769233),
 'R2': 0.021782355455959745}

In [328]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

previous_results = pd.read_csv(
    results_file
)

previous_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668
9,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815


In [329]:
rf_results = pd.DataFrame(
    [
        rf_spo2_result,
        rf_hr_result,
        rf_all_result
    ]
)

all_results = pd.concat(
    [
        previous_results,
        rf_results
    ],
    ignore_index=True
)

all_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668
41,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
38,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [330]:
all_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
20,Random Forest - SpO2 History,0.425950,1.415256,0.049668
26,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [331]:
all_results.to_csv(
    results_file,
    index=False
)

print("Random Forest results saved!")

Random Forest results saved!


In [332]:
error_analysis = common_val_df[
    [
        "caseid",
        "minute",
        "spo2",
        "future_spo2_5min"
    ]
].copy()

error_analysis["baseline_error"] = (
    (
        error_analysis["future_spo2_5min"]
        -
        error_analysis["spo2"]
    )
    .abs()
)

error_analysis.sort_values(
    "baseline_error",
    ascending=False
).head(20)

,caseid,minute,spo2,future_spo2_5min,baseline_error
618,7,1,62.0,99.0,37.0
11331,74,321,100.0,68.0,32.0
619,7,2,73.0,100.0,27.0
3687,22,229,100.0,83.0,17.0
865,7,248,100.0,84.5,15.5
3683,22,225,100.0,85.0,15.0
11336,74,326,68.0,80.0,12.0
3458,22,0,94.0,100.0,6.0
4663,28,131,93.0,99.0,6.0
4608,28,76,100.0,94.0,6.0


In [333]:
all_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668
41,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
38,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [334]:
all_results.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668
14,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [335]:
all_results.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668
14,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [336]:
[
    col
    for col in df.columns
    if "future" in col.lower()
]

['future_spo2_5min']

In [337]:
all_results.to_csv(
    results_file,
    index=False
)

print("Random Forest results saved!")

Random Forest results saved!


In [338]:
all_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668
41,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
38,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [339]:
all_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
20,Random Forest - SpO2 History,0.425950,1.415256,0.049668
26,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [340]:
all_results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
44,Random Forest - SpO2 History,0.425950,1.415256,0.049668
29,Random Forest - SpO2 History,0.425950,1.415256,0.049668
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
35,Random Forest - SpO2 History,0.425950,1.415256,0.049668
14,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Random Forest - SpO2 History,0.425950,1.415256,0.049668


In [341]:
all_results["Model"].value_counts()

Model
Random Forest - SpO2 History        15
Random Forest - SpO2 + HR           15
Random Forest - All Vitals          15
Persistence Baseline                 1
Linear Regression - SpO2 History     1
Ridge - SpO2 History                 1
Ridge - SpO2 + HR                    1
Ridge - All Vitals                   1
Name: count, dtype: int64

In [342]:
all_results = (
    all_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

all_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782


In [343]:
all_results.sort_values("MAE")

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
4,Ridge - All Vitals,0.512447,1.384467,0.090568


In [344]:
all_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
0,Persistence Baseline,0.266876,1.657234,-0.303086


In [345]:
all_results.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
0,Persistence Baseline,0.266876,1.657234,-0.303086
